[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ichristov/viscoelastic-startup/blob/main/notebooks/christov_2013_jeffreys_couette.ipynb)

# LEARNING OBJECTIVE

After exploring this notebook, you will be able to regenerate, from scratch, the correct and the erroneous ("textbook") eigenfunction-expansion solutions for the start-up of plane Couette flow of a Jeffreys ("Oldroyd-B") fluid given by [Christov (2013)](https://arxiv.org/abs/1305.5999), verify them against a numerical inversion of the Laplace transform and a finite-difference solution, and reproduce Fig. 1 of that paper.

# PRELIMINARIES

This notebook uses [NumPy](https://numpy.org) (Harris _et al._, 2020), [SciPy](https://scipy.org) (Virtanen _et al._, 2020), [SymPy](https://www.sympy.org) (Meurer _et al._, 2017), [Matplotlib](https://matplotlib.org) (Hunter, 2007) and [mpmath](https://mpmath.org) (mpmath development team, 2023).

[run the next cell to setup Python environment customizations and load packages]

In [ ]:
# interactive plots setup
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

# sympy setup
import sympy as sp
sp.init_printing()
from sympy.vector import *

# plotting customizations
from matplotlib import colormaps, animation, rc
from matplotlib import pyplot as plt
size=16
params = {'legend.fontsize': 'large',
#          'figure.figsize': (20,8),
          'axes.labelsize': size,
          'axes.titlesize': size,
          'xtick.labelsize': size*0.875,
          'ytick.labelsize': size*0.875,
          'axes.titlepad': 25}
plt.rcParams.update(params)
%matplotlib inline

# for animations
from IPython.display import HTML

# numerics
import numpy as np
from scipy.linalg import solve_banded
import mpmath as mp
import os

# for Colab only: to save plots as files and download them
#from google.colab import files

## Credit

Initial version of this notebook written by [Ivan C. Christov](http://christov.tmnt-lab.org), Purdue University.

Reproducing results of I. C. Christov.

# THE PROBLEM

The rheology is Eq. (2) of Christov (2013), where $\sigma$ is the shear stress and $\dot\gamma = \partial u/\partial y$:
$$
    \sigma + \tau \frac{\partial \sigma}{\partial t} = \dot\gamma + \alpha \frac{\partial \dot\gamma}{\partial t}.
$$
Combined with $\partial u/\partial t = \partial\sigma/\partial y$, the start-up of plane Couette flow is the initial-boundary-value problem (IBVP) of Eq. (3):
\begin{align*}
    \left(1 + \tau\frac{\partial}{\partial t}\right)\frac{\partial u}{\partial t} &= \frac{\partial}{\partial y} \left(1 + \alpha\frac{\partial}{\partial t}\right)\frac{\partial u}{\partial y},\qquad (y,t) \in (0,1)\times(0,\infty),\\
    u(y,0) &= \frac{\partial u}{\partial t}(y,0) = 0,\qquad 0<y<1,\\
    u(0,t) &= 1,\qquad t > 0,\\
    u(1,t) &= 0,\qquad t > 0.
\end{align*}
The dimensionless variables are
$$
    u = \frac{u^\star}{U_0},\qquad y=\frac{y^\star}{d},\qquad t=\frac{t^\star}{d^2/\nu},\qquad \tau = \frac{\lambda_1}{d^2/\nu},\qquad \alpha = \frac{\lambda_2}{d^2/\nu},
$$
where $d$ is the gap, $U_0$ the velocity of the bottom plate, $\nu$ the kinematic viscosity, and $0<\lambda_2<\lambda_1$ the retardation and relaxation times. The steady state, Eq. (4), is $u_\mathrm{ss}(y) = 1-y$. Fig. 1 of the paper uses $\tau = 1$ and $\alpha = 0.5$.

[pedagogy: one line on why $u(0,t)=1$ for $t>0$ really means $u(0,t)=H(t)$, the start-up jump]

# THE ERRONEOUS SOLUTION

> **⚠ ERRONEOUS SOLUTION**: the "textbook" eigenfunction expansion, Eqs. (15)&ndash;(16) of Christov (2013), obtained by writing $u = v + u_\mathrm{ss}$ and imposing $v(y,0) = -u_\mathrm{ss}(y)$. It is shown below to be wrong for $\alpha > 0$.

$$
    u(y,t) = (1 - y) - \frac{2}{\pi}\sum_{n=1}^\infty \exp\left(-\frac{1+\lambda_n\alpha}{2\tau}t\right) A_n(t) \frac{\sin(n\pi y)}{n},
$$
$$
    A_n(t) = \begin{cases}
    \dfrac{1+\lambda_n\alpha}{\sqrt{\Delta_n}}\sinh\left(\dfrac{t}{2\tau}\sqrt{\Delta_n}\right) + \cosh\left(\dfrac{t}{2\tau}\sqrt{\Delta_n}\right), &\Delta_n > 0,\\[3mm]
    \dfrac{1+\lambda_n\alpha}{2\tau}t + 1, &\Delta_n = 0,\\[3mm]
    \dfrac{1+\lambda_n\alpha}{\sqrt{|\Delta_n|}}\sin\left(\dfrac{t}{2\tau}\sqrt{|\Delta_n|}\right) + \cos\left(\dfrac{t}{2\tau}\sqrt{|\Delta_n|}\right), &\Delta_n < 0,
    \end{cases}
$$
with $\lambda_n = n^2\pi^2$ and $\Delta_n = (1+\lambda_n\alpha)^2 - 4\lambda_n\tau$.

In [ ]:
# WRONG, as printed: Eqs. (15)-(16) of Christov (2013), textbook eigenfunction series
def damped_A(t, lam, tau, alpha, c):
    """exp(-(1+lam*alpha)t/(2tau))*A_n(t) of Eqs. (16) and (43), with prefactor c = 1 +/- lam*alpha."""
    b = (1 + lam*alpha)/(2*tau)
    D = (1 + lam*alpha)**2 - 4*lam*tau
    if D > 0:
        # write sinh and cosh as exponentials, so that the large modes do not overflow
        sD = np.sqrt(D)
        e1 = np.exp((-b + sD/(2*tau))*t)
        e2 = np.exp((-b - sD/(2*tau))*t)
        return 0.5*c/sD*(e1 - e2) + 0.5*(e1 + e2)
    elif D == 0:
        return np.exp(-b*t)*(c/(2*tau)*t + 1)
    else:
        sD = np.sqrt(-D)
        return np.exp(-b*t)*(c/sD*np.sin(t*sD/(2*tau)) + np.cos(t*sD/(2*tau)))

def wrong_textbook_15(y, t, tau, alpha, nterms=50):
    """Erroneous solution, as printed: Eqs. (15)-(16) of Christov (2013)."""
    u = 1 - y
    for n in range(1, nterms+1):
        lam = (n*np.pi)**2
        u = u - 2/np.pi*damped_A(t, lam, tau, alpha, 1 + lam*alpha)*np.sin(n*np.pi*y)/n
    return u

# THE CORRECT SOLUTION

Writing $v = u - H(t)u_\mathrm{ss}$ and expanding $v = \sum_n a_n(t)\sin(n\pi y)$, [Duhamel's principle](https://en.wikipedia.org/wiki/Duhamel%27s_principle) gives Eq. (37),
$$
    \left(1 + \tau\frac{\mathrm{d}}{\mathrm{d}t}\right)\frac{\mathrm{d}a_n}{\mathrm{d}t} = -\lambda_n\left(1 + \alpha\frac{\mathrm{d}}{\mathrm{d}t}\right) a_n - \frac{2}{n\pi}[\delta(t) + \tau\delta'(t)],\qquad a_n(0) = \frac{\mathrm{d}a_n}{\mathrm{d}t}(0) = 0,
$$
whose [Laplace transform](https://en.wikipedia.org/wiki/Laplace_transform) is Eq. (38),
$$
    (1+\tau s)s \bar{a}_n = -\lambda_n(1+\alpha s)\bar{a}_n-\frac{2}{n\pi}(1+\tau s),
$$
so that, Eq. (41),
$$
    \bar{a}_n(s) = -\frac{2(1+\tau s)}{n\pi [(1+\tau s)s + \lambda_n(1+\alpha s)]}.
$$
The textbook approach instead solves Eq. (12) without a source, with the initial conditions of Eq. (14), $a_n(0) = -2/(n\pi)$ and $\mathrm{d}a_n/\mathrm{d}t(0) = 0$; its transform is Eq. (40),
$$
    (1+\tau s)s \bar{a}_n = -\lambda_n(1+\alpha s)\bar{a}_n - \frac{2}{n\pi} (1 + \tau s+ \lambda_n \alpha).
$$
The next cell derives both transforms with SymPy. It should print Eq. (41), then the difference between the two solutions, which is proportional to $\lambda_n\alpha$ and so vanishes only for $\alpha=0$.

In [ ]:
# Eqs. (37), (38), (40), (41) of Christov (2013): Laplace transforms of the two ODEs for a_n(t)
t = sp.Symbol('t', real=True)   # not positive=True, otherwise SymPy sets DiracDelta(t) = 0
s = sp.Symbol('s', positive=True)
tau, alpha, lam = sp.symbols('tau alpha lambda_n', positive=True)
n = sp.Symbol('n', integer=True, positive=True)
a = sp.Function('a')
abar, a0, a1 = sp.symbols(r'\bar{a}_n a_n(0) a_n^{\prime}(0)')

def laplace(expr):
    """Laplace transform in t, with abar, a_n(0), a_n'(0) as symbols."""
    L = sp.laplace_transform(expr, t, s, noconds=True)
    L = L.subs(sp.LaplaceTransform(a(t), t, s), abar)
    L = L.subs(sp.Subs(sp.Derivative(a(t), t), t, 0), a1).subs(a(0), a0)
    # SymPy leaves L{delta'(t)} unevaluated; it equals s (transform taken from t = 0^-)
    L = L.subs(sp.LaplaceTransform(sp.DiracDelta(t, 1), t, s), s)
    return sp.expand(L)

ode = (a(t).diff(t) + tau*a(t).diff(t, 2)) + lam*(a(t) + alpha*a(t).diff(t))
source = 2/(n*sp.pi)*(sp.DiracDelta(t) + tau*sp.DiracDelta(t).diff(t))

eq38 = laplace(ode + source).subs({a0: 0, a1: 0})
eq40 = laplace(ode).subs({a0: -2/(n*sp.pi), a1: 0})
abar_correct = sp.solve(eq38, abar)[0]
abar_wrong = sp.solve(eq40, abar)[0]
display(sp.Eq(abar, sp.factor(abar_correct)))
display(sp.Eq(sp.Symbol(r'\bar{a}_n^\mathrm{wrong} - \bar{a}_n'), sp.factor(abar_wrong - abar_correct)))

In [ ]:
# Eqs. (41) and (43) of Christov (2013): the Laplace transform of a_n(t) from Eq. (43) is Eq. (41)
# (checked for Delta_n > 0; the other two cases follow by continuation in Delta_n)
Delta = sp.Symbol('Delta_n', positive=True)
b = (1 + lam*alpha)/(2*tau)
c = sp.sqrt(Delta)/(2*tau)
a_n = -2/(n*sp.pi)*sp.exp(-b*t)*((1 - lam*alpha)/sp.sqrt(Delta)*sp.sinh(c*t) + sp.cosh(c*t))
L_a_n = sp.laplace_transform(a_n, t, s, noconds=True)
sp.simplify((L_a_n - abar_correct).subs(Delta, (1 + lam*alpha)**2 - 4*lam*tau))

Inverting Eq. (41) gives the correct solution, Eqs. (42)&ndash;(43) of Christov (2013), where $H(t)$ is the [Heaviside step function](https://en.wikipedia.org/wiki/Heaviside_step_function):
$$
    u(y,t) = H(t)\left[(1 - y) - \frac{2}{\pi}\sum_{n=1}^\infty \exp\left(-\frac{1+\lambda_n\alpha}{2\tau}t\right) A_n(t) \frac{\sin(n\pi y)}{n}\right],
$$
$$
    A_n(t) = \begin{cases}
    \dfrac{1-\lambda_n\alpha}{\sqrt{\Delta_n}}\sinh\left(\dfrac{t}{2\tau}\sqrt{\Delta_n}\right) + \cosh\left(\dfrac{t}{2\tau}\sqrt{\Delta_n}\right), &\Delta_n > 0,\\[3mm]
    \dfrac{1-\lambda_n\alpha}{2\tau}t + 1, &\Delta_n = 0,\\[3mm]
    \dfrac{1-\lambda_n\alpha}{\sqrt{|\Delta_n|}}\sin\left(\dfrac{t}{2\tau}\sqrt{|\Delta_n|}\right) + \cos\left(\dfrac{t}{2\tau}\sqrt{|\Delta_n|}\right), &\Delta_n < 0.
    \end{cases}
$$
The only change from Eqs. (15)&ndash;(16) is $+\lambda_n\alpha \mapsto -\lambda_n\alpha$ in the prefactors.

In [ ]:
# Eqs. (42)-(43) of Christov (2013): correct series
def unsteadyv(y, t, tau, alpha, nterms=25):
    """Correct solution, Eqs. (42)-(43) of Christov (2013)."""
    if t <= 0:
        return 0*y
    u = 1 - y
    for n in range(1, nterms+1):
        lam = (n*np.pi)**2
        u = u - 2/np.pi*damped_A(t, lam, tau, alpha, 1 - lam*alpha)*np.sin(n*np.pi*y)/n
    return u

[pedagogy: one line on why the source is $\delta(t) + \tau\delta^\prime(t)$, the time derivative of the jump $H(t)u_\mathrm{ss}(y)$ acted on by $1+\tau\partial_t$]

# AN INDEPENDENT CHECK

The Laplace transform of the IBVP, Eq. (21) of Christov (2013),
$$
    \frac{s(1+\tau s)}{1+\alpha s}\bar{u} = \frac{\partial^2 \bar{u}}{\partial y^2},\qquad \bar{u}(0,s) = \frac{1}{s},\qquad \bar{u}(1,s) = 0,
$$
has the solution, Eq. (22),
$$
    \bar{u}(y,s) = \frac{\sinh\left[\sqrt{\zeta(s)}(1-y)\right]}{s \sinh\left[\sqrt{\zeta(s)}\right]},\qquad \zeta(s) := \frac{s(1+\tau s)}{1+\alpha s}.
$$
The next cell checks this with SymPy; it should print three zeros.

In [ ]:
# Eqs. (21)-(22) of Christov (2013): Eq. (22) solves the subsidiary boundary-value problem
y = sp.Symbol('y')
zeta = s*(1 + tau*s)/(1 + alpha*s)
ubar_sym = sp.sinh(sp.sqrt(zeta)*(1 - y))/(s*sp.sinh(sp.sqrt(zeta)))
(sp.simplify(ubar_sym.diff(y, 2) - zeta*ubar_sym), sp.simplify(ubar_sym.subs(y, 0) - 1/s), ubar_sym.subs(y, 1))

Eq. (22) is inverted numerically by Tzou's Riemann sum, Eq. (44) of Christov (2013) (from Sect. 2.5.1 of Tzou (1997)),
$$
    u(y,t) \approx \frac{\mathrm{e}^{4.7}}{t}\left\{\frac{1}{2}\bar{u}\left(y,\frac{4.7}{t}\right)+\mathrm{Re}\left[ \sum_{m=1}^{M} (-1)^m \bar{u}\left(y,\frac{4.7+\mathrm{i} m \pi}{t}\right)\right] \right\},
$$
as in the paper (which used _Mathematica_), and, as a more accurate cross-check, by the algorithm of de Hoog _et al._ (1982) in [mpmath's `invertlaplace`](https://mpmath.org/doc/current/calculus/inverselaplace.html).

In [ ]:
# Eqs. (22) and (44) of Christov (2013): numerical inversion of the Laplace transform
def ubar(y, s, tau, alpha):
    """Eq. (22) of Christov (2013), written with exp(-z y) so that it does not overflow for large |s|."""
    z = np.sqrt(s*(1 + tau*s)/(1 + alpha*s))
    return np.exp(-z*y)*(1 - np.exp(-2*z*(1 - y)))/(s*(1 - np.exp(-2*z)))

def tzou_inverse(Fhat, t, M=10000):
    """Tzou's Riemann-sum inversion, Eq. (44) of Christov (2013)."""
    total = 0.5*Fhat(4.7/t).real
    for m in range(1, M+1):
        total = total + ((-1)**m*Fhat((4.7 + 1j*m*np.pi)/t)).real
    return np.exp(4.7)/t*total

def ubar_mp(s, y, tau, alpha):
    """Eq. (22) of Christov (2013) in mpmath arithmetic."""
    z = mp.sqrt(s*(1 + tau*s)/(1 + alpha*s))
    return mp.sinh(z*(1 - y))/(s*mp.sinh(z))

def dehoog_inverse(y, t, tau, alpha):
    """Eq. (22) of Christov (2013) inverted by mpmath's de Hoog algorithm, point by point in y."""
    # preferred over tzou_inverse: about 10 correct digits, while Tzou's formula
    # cannot do better than about exp(-9.4) = 8e-5, however large M is
    u = np.zeros_like(y)
    for j in range(len(y)):
        u[j] = float(mp.invertlaplace(lambda s: ubar_mp(s, y[j], tau, alpha), t, method='dehoog'))
    return u

The finite-difference check uses the solver `startup_fd` in polymer-stress form. With time scaled by $\lambda_1$ (so $t_\mathrm{fd} = t/\tau$), ${\rm Re} = 1/\tau$, $\kappa = \alpha/\tau$, and the moving plate at $x = 1-y = 1$, the polymer stress $S = \sigma - \kappa\,\partial v/\partial x$ obeys
\begin{align*}
    {\rm Re}\,\frac{\partial v}{\partial t_\mathrm{fd}} &= \frac{\partial S}{\partial x} + \kappa\frac{\partial^2 v}{\partial x^2},\\
    \frac{\partial S}{\partial t_\mathrm{fd}} + S &= (1-\kappa)\,\frac{\partial v}{\partial x},
\end{align*}
with $v$ at the nodes and $S$ at the cell centers, Crank&ndash;Nicolson for the diffusion (backward Euler for the first four steps, a Rannacher start-up) and the trapezoidal rule for $S$.

In [ ]:
# Finite differences for Eq. (3) of Christov (2013): startup_fd, polymer-stress form
def startup_fd(M=400, kappa=0.5, Re=1.0, tmax=1.0, L=1.0, dt=None, n_rannacher=4, n_iter=3):
    """Start-up plane Couette flow of a Jeffreys fluid, (v, S) form; time scaled by lambda_1, plate at x = L."""
    dx = L/M
    if dt is None:
        dt = 0.5*dx*min(1.0, np.sqrt(Re))
    nsteps = int(np.ceil(tmax/dt))
    dt = tmax/nsteps
    x = np.linspace(0, L, M+1)
    v = np.zeros(M+1)
    S = np.zeros(M)
    v[-1] = 1.0
    # tridiagonal matrices for Crank-Nicolson (theta = 1/2) and backward Euler (theta = 1)
    r_cn = 0.5*kappa*dt/(Re*dx**2)
    ab_cn = np.zeros((3, M-1))
    ab_cn[0, 1:] = -r_cn
    ab_cn[1, :] = 1 + 2*r_cn
    ab_cn[2, :-1] = -r_cn
    r_be = kappa*dt/(Re*dx**2)
    ab_be = np.zeros((3, M-1))
    ab_be[0, 1:] = -r_be
    ab_be[1, :] = 1 + 2*r_be
    ab_be[2, :-1] = -r_be
    for n in range(nsteps):
        g = np.diff(v)/dx
        dS = -S + (1 - kappa)*g
        Sp = S + dt*dS
        if n < n_rannacher:
            r = r_be
            ab = ab_be
        else:
            r = r_cn
            ab = ab_cn
        vb = v.copy()
        # iterate the trapezoidal stress update and the implicit velocity update a few times
        for it in range(n_iter):
            Sm = 0.5*(S + Sp)
            rhs = v[1:-1] + dt/Re*np.diff(Sm)/dx
            if n >= n_rannacher:
                rhs = rhs + r*(v[2:] - 2*v[1:-1] + v[:-2])
            rhs[0] = rhs[0] + r*vb[0]
            rhs[-1] = rhs[-1] + r*vb[-1]
            vb[1:-1] = solve_banded((1, 1), ab, rhs)
            gp = np.diff(vb)/dx
            Sp = S + 0.5*dt*(dS + (-Sp + (1 - kappa)*gp))
        S = Sp
        v = vb
    return x, v

def fd_profile(t, tau, alpha, M=400):
    """u(y,t) of Eq. (3) of Christov (2013) from startup_fd, mapped back to y = 1 - x."""
    x, v = startup_fd(M=M, kappa=alpha/tau, Re=1/tau, tmax=t/tau)
    return 1 - x, v

The next cell prints all the checks:

* the Newtonian limit $\alpha=\tau$ against the Newtonian solution, Eq. (35) of Christov (2013),
$$
    u(y,t) = H(t)\left[ (1 - y) - \frac{2}{\pi}\sum_{n=1}^\infty \exp\left(-n^2\pi^2 t\right)\frac{\sin(n\pi y)}{n}\right],
$$
  and the Maxwell limit $\alpha = 0$, where the wrong and correct series coincide (both differences should be at round-off level);
* the correct series against Tzou's inversion (should be below $10^{-4}$), de Hoog's inversion (should be about $10^{-10}$) and finite differences, and the size of the error of the wrong series;
* convergence tables for the series, Tzou's $M$, and the finite-difference grid. The finite differences converge at first order for the Jeffreys fluid, because the initial and boundary data are incompatible at $(y,t)=(0,0)$ and the polymer stress, which does not diffuse, keeps the resulting error; the same solver is second order in the Newtonian limit.

In [ ]:
# Checks: limits, agreement, and convergence for Christov (2013)
tau, alpha = 1.0, 0.5
times = [0.01, 0.1, 1.0]
y = np.linspace(0, 1, 21)[1:-1]   # interior points

print('LIMITS (should be at round-off level)')
t = 0.05
newtonian = 1 - y
for n in range(1, 401):
    newtonian = newtonian - 2/np.pi*np.exp(-(n*np.pi)**2*t)*np.sin(n*np.pi*y)/n
print(f'  Newtonian, alpha = tau = 1, t = {t}: max|Eq. (42) - Eq. (35)| = {np.abs(unsteadyv(y, t, 1.0, 1.0, 400) - newtonian).max():.1e}')
print(f'  Maxwell, alpha = 0, tau = 1, t = 0.3: max|Eq. (15) - Eq. (42)| = {np.abs(wrong_textbook_15(y, 0.3, 1.0, 0.0, 400) - unsteadyv(y, 0.3, 1.0, 0.0, 400)).max():.1e}')

print(f'\nCORRECT SERIES (4000 terms) VERSUS INDEPENDENT SOLUTIONS, tau = {tau}, alpha = {alpha}, max over 0 < y < 1')
print('     t   Tzou (M=10^4)   de Hoog   FD (M=400)   wrong - correct')
exact = {}
for t in times:
    exact[t] = unsteadyv(y, t, tau, alpha, 4000)
    tz = tzou_inverse(lambda s: ubar(y, s, tau, alpha), t, 10000)
    dh = dehoog_inverse(y, t, tau, alpha)
    yf, uf = fd_profile(t, tau, alpha, 400)
    fd = np.interp(y, yf[::-1], uf[::-1])
    wr = wrong_textbook_15(y, t, tau, alpha, 4000)
    print(f'  {t:5.2f}   {np.abs(tz - exact[t]).max():.1e}         {np.abs(dh - exact[t]).max():.1e}   {np.abs(fd - exact[t]).max():.1e}      {np.abs(wr - exact[t]).max():.2f}')

print('\nCONVERGENCE OF THE CORRECT SERIES (versus 4000 terms)')
print('  terms   ' + '   '.join(f't = {t:<5}' for t in times))
for nterms in [25, 50, 100, 400]:
    row = f'  {nterms:5d}'
    for t in times:
        row = row + f'   {np.abs(unsteadyv(y, t, tau, alpha, nterms) - exact[t]).max():.1e}  '
    print(row)

print('\nCONVERGENCE OF TZOU\'S INVERSION IN M, t = 0.1 (floor about 8e-5)')
for M in [100, 1000, 10000]:
    print(f'  M = {M:5d}: {np.abs(tzou_inverse(lambda s: ubar(y, s, tau, alpha), 0.1, M) - exact[0.1]).max():.1e}')

print('\nCONVERGENCE OF THE FINITE DIFFERENCES (dt refined with dx), max error and observed order')
cases = [('Jeffreys, tau = 1, alpha = 0.5', 1.0, 0.5), ('Newtonian limit, tau = alpha = 1', 1.0, 1.0)]
for label, tau_c, alpha_c in cases:
    print('  ' + label)
    previous = None
    for M in [50, 100, 200, 400, 800]:
        row = f'    M = {M:4d}:'
        errors = []
        for t in times:
            yf, uf = fd_profile(t, tau_c, alpha_c, M)
            errors.append(np.abs(uf - unsteadyv(yf, t, tau_c, alpha_c, 4000))[1:-1].max())
        for k in range(len(times)):
            row = row + f'  {errors[k]:.1e}'
            if previous is not None:
                row = row + f' ({np.log2(previous[k]/errors[k]):.2f})'
        print(row)
        previous = errors

This is Fig. 1 of Christov (2013): $\tau = 1$, $\alpha = 0.5$, $t = 0.01$, $0.1$, $1$ (dark to bright), with 50 terms of the wrong series (dashed), 25 terms of the correct series (solid), Tzou's inversion with $M=10^4$ (open symbols), finite differences (filled symbols) and the steady state (thin line).

In [ ]:
# Fig. 1 of Christov (2013): wrong series, correct series, Tzou inversion, finite differences
tau, alpha = 1.0, 0.5
times = [0.01, 0.1, 1.0]
y = np.linspace(0, 1, 401)
y_tzou = np.linspace(0.05, 0.95, 10)
cmap = colormaps['viridis']

fig, ax = plt.subplots(figsize=(7, 7), tight_layout=True)
ax.plot(1 - y, y, color='gray', linewidth=1)
for t in times:
    color = cmap(0.6*np.log(t/times[0])/np.log(times[-1]/times[0]))
    ax.plot(wrong_textbook_15(y, t, tau, alpha, 50), y, color=color, linewidth=2, linestyle='dashed')
    ax.plot(unsteadyv(y, t, tau, alpha, 25), y, color=color, linewidth=3)
    ax.plot(tzou_inverse(lambda s: ubar(y_tzou, s, tau, alpha), t, 10000), y_tzou,
            'o', color=color, markerfacecolor='white', markersize=8)
    yf, uf = fd_profile(t, tau, alpha, 400)
    ax.plot(uf[40:-1:40], yf[40:-1:40], 'o', color=color, markersize=6)   # y = 0.1, ..., 0.9, between the Tzou points
    y_label = {0.01: 0.06, 0.1: 0.3, 1.0: 0.6}[t]
    ax.annotate(f'$t = {t:g}$', xy=(unsteadyv(np.array(y_label), t, tau, alpha, 400) + 0.03, y_label), color=color, fontsize=14)

# legend entries for the methods, in black
ax.plot([], [], color='black', linewidth=2, linestyle='dashed', label='wrong: Eqs. (15)-(16), 50 terms')
ax.plot([], [], color='black', linewidth=3, label='correct: Eqs. (42)-(43), 25 terms')
ax.plot([], [], 'o', color='black', markerfacecolor='white', label='Tzou, Eq. (44), $M=10^4$')
ax.plot([], [], 'o', color='black', label='finite differences')
ax.plot([], [], color='gray', linewidth=1, label='steady state, Eq. (4)')
ax.set_xlabel('$u$')
ax.set_ylabel('$y$')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1)
ax.grid(alpha=0.5, linestyle='dotted')
ax.legend(framealpha=1, shadow=True, fontsize=11, loc='upper right')
os.makedirs('../figures', exist_ok=True)
fig.savefig('../figures/christov_2013_fig1.png', dpi=200, bbox_inches='tight')

[pedagogy: one line on what "not causal" means here: the textbook initial condition encodes the whole future boundary input at $t=0$]

In [ ]:
# Eqs. (15)-(16) and (42)-(43) of Christov (2013): interactive comparison
def plot_func(tau, alpha, t):
    y = np.linspace(0, 1, 41)
    u = unsteadyv(y, t, tau, alpha, 200)
    u_wrong = wrong_textbook_15(y, t, tau, alpha, 200)

    fig, ax = plt.subplots(tight_layout=True)
    ax.set_ylabel('$y$')
    ax.set_ylim(0, 1)
    ax.set_xlabel('$u$')
    ax.set_xlim(-0.25, 1.25)
    ax.plot(1 - y, y, color='gray', linewidth=1)
    ax.plot(u, y, color='darkblue', linewidth=4, label=f'correct, $t={t:.3f}$')
    ax.quiver(0*y[::2], y[::2], u[::2], 0*u[::2], scale=1., scale_units='xy', color='blue', zorder=2)
    ax.plot(u_wrong, y, color='#D55E00', linewidth=2, linestyle='dashed', label='wrong: Eqs. (15)-(16)')
    ax.legend(framealpha=1, shadow=True)
    ax.grid(alpha=0.5, linestyle='dotted')
    plt.show()

interact(plot_func,
         tau=widgets.FloatSlider(value=1.0, min=0.05, max=2, step=0.05, description='Relax., tau'),
         alpha=widgets.FloatSlider(value=0.5, min=0.0, max=2, step=0.05, description='Retard., alpha'),
         t=widgets.FloatSlider(value=0.1, min=0.001, max=2, step=0.001, readout_format='.3f', description='Time, t'));

In [ ]:
# Eqs. (15)-(16) and (42)-(43) of Christov (2013): animation of the start-up
tau, alpha = 1.0, 0.5
y = np.linspace(0, 1, 41)
u = unsteadyv(y, 0.002, tau, alpha, 200)

fig, ax = plt.subplots(tight_layout=True)
ax.set_ylabel('$y$')
ax.set_ylim(0, 1)
ax.set_xlabel('$u$')
ax.set_xlim(-0.25, 1.25)
ax.grid(alpha=0.5, linestyle='dotted')
plt.close()

ax.plot(1 - y, y, color='gray', linewidth=1)
line, = ax.plot(u, y, color='darkblue', linewidth=4, label='correct')
line_wrong, = ax.plot(wrong_textbook_15(y, 0.002, tau, alpha, 200), y, color='#D55E00',
                      linewidth=2, linestyle='dashed', label='wrong: Eqs. (15)-(16)')
quiv = ax.quiver(0*y[::2], y[::2], u[::2], 0*u[::2], scale=1., scale_units='xy', color='blue', zorder=2)
time_label = ax.annotate('$t=0.002$', xy=(0.7, 0.85), bbox=dict(boxstyle="round", facecolor="white"))
ax.legend(framealpha=1, shadow=True, loc='lower right')

# number of animation frames
numframes = 100
tmax = 1.0

# animation function called sequentially by `FuncAnimation' below
def animate(i):
    # the i passed is the frame counter, rescaled to get the time
    tstar = (i + 1)/numframes*tmax
    u = unsteadyv(y, tstar, tau, alpha, 200)
    line.set_data(u, y)
    line_wrong.set_data(wrong_textbook_15(y, tstar, tau, alpha, 200), y)
    quiv.set_UVC(u[::2], 0*u[::2])
    time_label.set_text(f'$t={tstar:.3f}$')
    return (line, line_wrong, quiv, time_label)

anim = animation.FuncAnimation(fig, animate, frames=numframes, interval=100, blit=True, repeat=False)

In [ ]:
# this is necessary to get the animation to work on Google's Colab
rc('animation', html='jshtml')
anim

In [ ]:
# To save the animation, use something like
# anim.save('christov_2013_startup.mp4')

# WHAT PROBLEM DOES THE WRONG SOLUTION ACTUALLY SOLVE?

In the Laplace domain, the coefficient of $\sin(n\pi y)$ in $\bar{u}$ is $2/(n\pi s) + \bar{a}_n$. For every $n$, the textbook coefficient (from Eq. (40)) is the correct one (from Eq. (38)) divided by $1+\alpha s$. So the wrong series solves the IBVP (3) with the plate started not impulsively but ramped on the retardation time,
$$
    u(0,t) = \left(1 - \mathrm{e}^{-t/\alpha}\right)H(t),\qquad \bar{u}_\mathrm{wrong}(y,s) = \frac{\bar{u}(y,s)}{1+\alpha s}.
$$
This identification is shown here; it is not stated in Christov (2013). The next cell should print $1/(\alpha s + 1)$.

In [ ]:
# Eqs. (38) and (40) of Christov (2013): ratio of the textbook to the correct Laplace-domain mode
n = sp.Symbol('n', integer=True, positive=True)   # n was reused as a loop index above
mode_correct = 2/(n*sp.pi*s) + abar_correct
mode_wrong = 2/(n*sp.pi*s) + abar_wrong
sp.simplify(mode_wrong/mode_correct)

The next cell compares the wrong series (20000 terms) with the de Hoog inversion of $\bar{u}(y,s)/(1+\alpha s)$; the difference should be limited only by the truncation of the series.

In [ ]:
# Wrong series of Eqs. (15)-(16) versus the ramped-plate solution, inverse of Eq. (22)/(1 + alpha s)
tau, alpha = 1.0, 0.5
y = np.array([0.25, 0.5, 0.75])
max_difference = 0.0
for t in [0.1, 0.5, 1.0]:
    u_wrong = wrong_textbook_15(y, t, tau, alpha, 20000)
    u_ramp = np.zeros_like(y)
    for j in range(len(y)):
        u_ramp[j] = float(mp.invertlaplace(lambda s: ubar_mp(s, y[j], tau, alpha)/(1 + alpha*s), t, method='dehoog'))
    max_difference = max(max_difference, np.abs(u_wrong - u_ramp).max())
print(f'max|wrong series - ramped-plate solution| = {max_difference:.1e}')

# ON YOUR OWN

On your own:

*   Set $\alpha = 0$ (Maxwell fluid) and $\alpha = \tau$ (Newtonian fluid) in the interactive plot, and explain from Eqs. (38) and (40) why the wrong series is right in these two cases.
*   Increase the number of terms of the wrong series at $t = 0.01$ and find the value it tends to as $y \to 0^+$.
*   Regenerate Fig. 1 for $\alpha/\tau = 0.05$ and $0.4$.

# REFERENCES

I. C. Christov, On a difficulty in the formulation of initial and boundary conditions for eigenfunction expansion solutions for the start-up of fluid flow, _Mech. Res. Commun._ **51** (2013) 86&ndash;92. [doi:10.1016/j.mechrescom.2013.05.005](https://doi.org/10.1016/j.mechrescom.2013.05.005); [arXiv:1305.5999](https://arxiv.org/abs/1305.5999)

F. R. de Hoog, J. H. Knight, A. N. Stokes, An improved method for numerical inversion of Laplace transforms, _SIAM J. Sci. Stat. Comput._ **3** (1982) 357&ndash;366. [doi:10.1137/0903022](https://doi.org/10.1137/0903022)

C. R. Harris, K. J. Millman, S. J. van der Walt, _et al._, Array programming with NumPy, _Nature_ **585** (2020) 357&ndash;362. [doi:10.1038/s41586-020-2649-2](https://doi.org/10.1038/s41586-020-2649-2)

J. D. Hunter, Matplotlib: A 2D graphics environment, _Comput. Sci. Eng._ **9** (2007) 90&ndash;95. [doi:10.1109/MCSE.2007.55](https://doi.org/10.1109/MCSE.2007.55)

A. Meurer, C. P. Smith, M. Paprocki, _et al._, SymPy: symbolic computing in Python, _PeerJ Comput. Sci._ **3** (2017) e103. [doi:10.7717/peerj-cs.103](https://doi.org/10.7717/peerj-cs.103)

The mpmath development team, _mpmath: a Python library for arbitrary-precision floating-point arithmetic_ (version 1.3.0), 2023. [mpmath.org](https://mpmath.org/)

D. Y. Tzou, _Macro- to Microscale Heat Transfer: The Lagging Behavior_, Taylor & Francis, Washington, DC, 1997.

P. Virtanen, R. Gommers, T. E. Oliphant, _et al._, SciPy 1.0: fundamental algorithms for scientific computing in Python, _Nat. Methods_ **17** (2020) 261&ndash;272. [doi:10.1038/s41592-019-0686-2](https://doi.org/10.1038/s41592-019-0686-2)